# 05 · Integrated vapor transport for each node

**Goal.** Compute vertically integrated water vapor transport (IVT) for the same days, composite
it by SOM node, and use it to ask *why* some flow regimes deliver more rain than others.

## Why IVT and not just 850 hPa wind

The SOM in notebook 04 classifies days by 850 hPa wind — a single level, and wind alone. That is
a good *classifier* (it captures the flow geometry) but an incomplete *explanation*: a strong
wind blowing dry air delivers no rain.

IVT combines both ingredients into the physically meaningful quantity — the total flux of water
vapor through a unit-width vertical column:

$$\mathbf{IVT} = \frac{1}{g}\int_{p_\text{top}}^{p_\text{sfc}} q\,\mathbf{V}\;dp
\qquad \left[\mathrm{kg\,m^{-1}\,s^{-1}}\right]$$

with $q$ specific humidity (kg/kg), $\mathbf{V} = (u, v)$ the horizontal wind (m/s), $p$ pressure
(Pa), and $g = 9.80665\ \mathrm{m\,s^{-2}}$.

Because the SOM never saw humidity, IVT is a **partially independent** view of the nodes. If the
wind-based classes differ systematically in IVT, that is evidence the classification is capturing
something dynamically real rather than an arbitrary partition.

## Integration limits

We integrate **300 → 1000 hPa** over 20 levels. Rationale:

- **1000 hPa bottom** — the lowest level in the ARCO-ERA5 pressure-level set. Below-ground levels
  over high terrain are handled by the terrain mask, as in notebooks 03–04.
- **300 hPa top** — the conventional cut in the atmospheric-river / IVT literature. Specific
  humidity above 300 hPa is smaller by orders of magnitude and contributes negligibly.
- **Uneven level spacing** — the ARCO-ERA5 set is denser near the surface (25 hPa steps below
  750 hPa, 50 hPa above), which is exactly where most of the moisture sits. We use trapezoidal
  integration on the actual pressure coordinates rather than assuming uniform $\Delta p$.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

import sys
sys.path.append("../src")
from config import (ARCO_STORE, DATA_DIR, FIG_DIR, NWSA_LAT, NWSA_LON_360,
                    NWSA_EXTENT, IVT_LEVELS, SNAPSHOT_HOUR, G)

assign = pd.read_csv(DATA_DIR / "som_node_assignments.csv", parse_dates=["date"])
terrain_mask = np.load(DATA_DIR / "terrain_mask_1500m.npy")

print(len(assign), "days;", len(IVT_LEVELS), "levels:", IVT_LEVELS)
print("g =", G, "m/s^2")

## 1. Compute IVT per day

The cost profile changes from notebook 03: 3 variables × 20 levels instead of 2 variables × 1
level. But the **chunk count is identical** — ARCO-ERA5 bundles all 37 levels into a single
chunk per variable per hour, so pulling 20 levels costs the same reads as pulling 1. Only the
per-chunk decompression and the transferred subset grow.

Integration uses `np.trapezoid`. Note the API: NumPy renamed `trapz` → `trapezoid` in 2.0, and
the old name is removed in recent versions — the shim below keeps the notebook running on both.

In [ ]:
# NumPy 2.0 renamed trapz -> trapezoid; support both
_trapz = getattr(np, "trapezoid", None) or np.trapz

def compute_ivt_day(ds, date, levels=IVT_LEVELS, hour=SNAPSHOT_HOUR):
    """IVT components for one day. Returns (ivt_u, ivt_v) in kg/m/s."""
    t = pd.Timestamp(date).normalize() + pd.Timedelta(hours=hour)
    sel = ds[["specific_humidity", "u_component_of_wind", "v_component_of_wind"]].sel(
        time=t, level=list(levels),
        latitude=slice(NWSA_LAT[1], NWSA_LAT[0]),
        longitude=slice(*NWSA_LON_360),
    ).load()

    q = sel["specific_humidity"].values          # (level, lat, lon), kg/kg
    u = sel["u_component_of_wind"].values
    v = sel["v_component_of_wind"].values
    p_pa = np.asarray(levels, dtype="float64") * 100.0    # hPa -> Pa, ascending

    # integrate along the level axis; ascending p means the result is positive downward-weighted
    return _trapz(q * u, p_pa, axis=0) / G, _trapz(q * v, p_pa, axis=0) / G

In [ ]:
ivt_path = DATA_DIR / "era5_ivt_heavy_enso.nc"

if ivt_path.exists():
    ivt = xr.open_dataset(ivt_path)
    print("loaded cached IVT file")
else:
    ds = xr.open_zarr(ARCO_STORE, chunks=None,
                      storage_options={"token": "anon"}, decode_timedelta=True)
    lat_ref = ds.latitude.sel(latitude=slice(NWSA_LAT[1], NWSA_LAT[0])).values
    lon_ref = ds.longitude.sel(longitude=slice(*NWSA_LON_360)).values

    iu, iv, used = [], [], []
    for i, d in enumerate(assign["date"].values):
        try:
            a, b = compute_ivt_day(ds, d)
        except Exception as exc:
            print(f"  SKIP {d}: {type(exc).__name__}: {exc}")
            continue
        iu.append(a); iv.append(b); used.append(pd.Timestamp(d).normalize())
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(assign)}", flush=True)

    IU, IV = np.stack(iu), np.stack(iv)
    ivt = xr.Dataset(
        {"ivt_u": (("time", "latitude", "longitude"), IU),
         "ivt_v": (("time", "latitude", "longitude"), IV),
         "ivt":   (("time", "latitude", "longitude"), np.sqrt(IU**2 + IV**2))},
        coords={"time": pd.DatetimeIndex(used), "latitude": lat_ref, "longitude": lon_ref},
    )
    ivt.attrs.update({
        "source": ARCO_STORE,
        "formula": "IVT = (1/g) * integral_300hPa^1000hPa q*V dp",
        "levels_hPa": list(IVT_LEVELS),
        "time_of_day": f"{SNAPSHOT_HOUR:02d}:00 UTC snapshot",
        "units": "kg m-1 s-1",
    })
    ivt.to_netcdf(ivt_path)
    print("wrote", ivt_path)

print(ivt.sizes)

## 2. Verify the magnitudes against known physical ranges

IVT is a good quantity to sanity-check because the literature gives firm reference values:

| Regime | \|IVT\| (kg m⁻¹ s⁻¹) |
|---|---|
| Dry subtropical subsidence | < 100 |
| Typical tropical background | 200–400 |
| Atmospheric-river threshold (mid-latitude convention) | ≥ 250 |
| Strong tropical moisture plume | 600–1000+ |

Values in the low thousands would indicate a unit error (a factor of 100 from hPa vs. Pa is the
classic one); values below ~10 would indicate the integration collapsed.

In [ ]:
mag = ivt.ivt.values
print("daily |IVT| statistics over all days and cells:")
for label, val in [("min", mag.min()), ("5th pct", np.percentile(mag, 5)),
                   ("median", np.median(mag)), ("95th pct", np.percentile(mag, 95)),
                   ("max", mag.max())]:
    print(f"  {label:9s} {val:8.1f} kg/m/s")
print("\nnon-finite fraction:", float((~np.isfinite(mag)).mean()))

# unit cross-check: recompute one day with an explicitly different formulation
d0 = assign["date"].iloc[0]
col_q_scale = float(np.nanmax(mag)) / float(np.nanmax(np.sqrt(
    ivt.ivt_u.values**2 + ivt.ivt_v.values**2)))
assert abs(col_q_scale - 1.0) < 1e-9, "ivt magnitude inconsistent with its components"
print("\nmagnitude/component consistency: OK")

## 3. Composite by node

Same alignment discipline as notebook 04 — match on date, never on position.

One subtlety worth stating explicitly, because it changes the answer: we composite the
**vector** components and then take the magnitude of the mean vector,

$$|\overline{\mathbf{IVT}}| = \sqrt{\bar{u}_{\rm IVT}^2 + \bar{v}_{\rm IVT}^2}$$

rather than averaging the daily magnitudes, $\overline{|\mathbf{IVT}|}$. The two differ whenever
transport *direction* varies within a node: the vector mean is smaller, and the gap between them
measures directional consistency. The vector mean is the right choice here because we want the
node's coherent, persistent transport — the part that survives averaging — not the typical
instantaneous intensity.

In [ ]:
ivt_dates = pd.to_datetime(ivt.time.values).normalize()
assert set(ivt_dates) <= set(assign["date"].dt.normalize()), "IVT has dates absent from assignments"

meta = assign.set_index(assign["date"].dt.normalize()).loc[ivt_dates].reset_index(drop=True)
node_id = meta["som_node"].values
counts = np.bincount(node_id, minlength=9)

n_nodes = 9
node_iu = np.zeros((n_nodes, *ivt.ivt_u.shape[1:]))
node_iv = np.zeros((n_nodes, *ivt.ivt_v.shape[1:]))
for k in range(n_nodes):
    m = node_id == k
    node_iu[k] = ivt.ivt_u.values[m].mean(axis=0)
    node_iv[k] = ivt.ivt_v.values[m].mean(axis=0)
node_mag = np.sqrt(node_iu**2 + node_iv**2)

# the two averaging choices, side by side
vec_mean = np.array([float(np.nanmean(np.where(terrain_mask, np.nan, node_mag[k]))) for k in range(9)])
scalar_mean = np.array([float(np.nanmean(np.where(terrain_mask[None], np.nan,
                        ivt.ivt.values[node_id == k]))) for k in range(9)])

print(pd.DataFrame({"n_days": counts,
                    "|mean vector|": vec_mean.round(1),
                    "mean |daily|": scalar_mean.round(1),
                    "directional_consistency": (vec_mean / scalar_mean).round(3)}).to_string())

The `directional_consistency` ratio is a free diagnostic: near 1 means every day in that node
transported moisture in nearly the same direction (a tight, coherent regime); well below 1 means
the node lumps together days with opposing transport, which is a hint that the node is a
catch-all rather than a regime — or that it is too small to be meaningful.

## 4. The terrain check, again — and why you must repeat it

The 1500 m mask was derived in notebook 03 for *wind*. IVT is a different quantity, so the check
has to be redone rather than assumed to carry over. It does not carry over for free: IVT
integrates over 20 levels, most of them well above ground, so one might reasonably expect it to
be less contaminated.

Run the same elevation stratification. The thing that identifies an artifact rather than a
physical signal is that it appears **identically in every node** — a real wind-shadow or
rain-shadow effect would vary between flow regimes.

In [ ]:
topo = xr.open_dataarray(DATA_DIR / "etopo2_nwsa.nc")
elev = topo.interp(lat=("latitude", ivt.latitude.values),
                   lon=("longitude", ivt.longitude.values - 360)).values

bands = [(-1e9, 0, "ocean"), (0, 500, "0-500 m"), (500, 1500, "500-1500 m"),
         (1500, 3000, "1500-3000 m"), (3000, 1e9, ">3000 m")]

tab = {}
for k in [0, 1, 4, 8]:
    tab[f"node {k}"] = {lab: round(float(node_mag[k][(elev > lo) & (elev <= hi)].mean()), 1)
                        for lo, hi, lab in bands if ((elev > lo) & (elev <= hi)).sum()}
print(pd.DataFrame(tab).to_string())

The monotonic decline with elevation, present in *every* node sampled, confirms the same
below-ground extrapolation problem affects IVT: the integral's lower levels are non-physical over
high terrain, so the column integral is biased low there regardless of the atmospheric state.
Mask it, and say so in the caption.

**Do not mistake the masked ridge for the real dry corridor.** There is also a genuine low-IVT
band over the *coastal lowlands* — at low elevation, unmasked — which is the signature of the
cold Humboldt Current suppressing column moisture just offshore and inland of the coast. That
one is physics and should be interpreted; the grey ridge is an artifact and should not.

In [ ]:
def plot_ivt_nodes(cu, cv, counts, terrain_mask, lon, lat, *,
                   cmap="ChaseSpectral", vmax=350, step=6, outfile=None):
    """Node grid of composite IVT. `cmap` defaults to cmweather's ChaseSpectral."""
    try:
        import cmweather  # noqa: F401  -- registers ChaseSpectral et al.
    except ImportError:
        cmap = "YlGnBu"
        print("cmweather not installed; falling back to YlGnBu")

    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    import geopandas as gpd
    from config import PROVINCES_GPKG

    proj = ccrs.PlateCarree()
    lon2d, lat2d = np.meshgrid(lon, lat)
    prov = gpd.read_file(PROVINCES_GPKG)
    norm = mpl.colors.Normalize(0, vmax)
    cm = plt.get_cmap(cmap)

    fig, axes = plt.subplots(3, 3, figsize=(10.5, 11.5), subplot_kw=dict(projection=proj),
                             gridspec_kw=dict(wspace=0.05, hspace=0.12))
    for k in range(9):
        ax = axes[k // 3, k % 3]
        ax.set_extent(NWSA_EXTENT, crs=proj)
        mag = np.sqrt(cu[k]**2 + cv[k]**2)
        ax.pcolormesh(lon2d, lat2d, np.where(terrain_mask, np.nan, mag),
                      cmap=cm, norm=norm, transform=proj, rasterized=True, shading="auto")
        ax.pcolormesh(lon2d, lat2d, np.where(terrain_mask, 1.0, np.nan),
                      cmap=mpl.colors.ListedColormap(["0.55"]), transform=proj,
                      rasterized=True, shading="auto")
        ax.quiver(lon2d[::step, ::step], lat2d[::step, ::step],
                  np.where(terrain_mask, np.nan, cu[k])[::step, ::step],
                  np.where(terrain_mask, np.nan, cv[k])[::step, ::step],
                  transform=proj, scale=3000, width=0.004, color="0.15")
        ax.coastlines(resolution="10m", linewidth=0.5, color="0.15")
        ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.4, color="0.15")
        prov.boundary.plot(ax=ax, linewidth=0.3, edgecolor="0.4", transform=proj)
        ax.set_title(f"Node {k} (n={counts[k]})", fontsize=9, pad=3)
        ax.set_xticks([]); ax.set_yticks([])

    cax = fig.add_axes([0.3, 0.055, 0.4, 0.015])
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cm), cax=cax,
                      orientation="horizontal", extend="max")
    cb.set_label("Integrated vapor transport, |IVT| (kg m$^{-1}$ s$^{-1}$)", fontsize=8.5, labelpad=6)
    cb.ax.tick_params(labelsize=7)
    fig.suptitle("Integrated vapor transport (300-1000 hPa) by 850 hPa wind SOM node",
                 fontsize=12, y=0.98)
    fig.text(0.5, 0.925,
             "Composite mean IVT VECTOR per node (not mean of daily magnitudes), 12:00 UTC ARCO-ERA5.\n"
             "Grey: terrain > 1500 m, masked -- below-surface pressure levels are non-physical extrapolations.",
             ha="center", fontsize=8, color="0.3")
    fig.subplots_adjust(left=0.02, right=0.98, top=0.87, bottom=0.10)
    if outfile:
        fig.savefig(outfile, dpi=200)
    return fig

fig = plot_ivt_nodes(node_iu, node_iv, counts, terrain_mask,
                     ivt.longitude.values - 360, ivt.latitude.values,
                     outfile=FIG_DIR / "som_ivt_nodes.png")

## 5. Does IVT explain the precipitation differences?

The payoff question. The SOM was built from wind alone; precipitation and IVT were never inputs.
If nodes with stronger moisture transport are also the wetter nodes, the classification is
capturing a physically coherent chain (flow regime → moisture delivery → rainfall).

Two caveats on interpreting the correlation below:

1. **n = 9.** A correlation over nine nodes is a weak statistic; treat it as a consistency
   check, not a result. Small nodes make it noisier still.
2. **Domain-mean IVT is a blunt predictor.** Rainfall over Guayas depends on transport
   *converging on and ascending over* the coastal Andes, not on the domain average. A node can
   have high domain-mean IVT directed away from the coast.

In [ ]:
node_summary = pd.read_csv(DATA_DIR / "som_node_summary.csv", index_col=0)
comp = node_summary.join(pd.DataFrame({
    "mean_ivt_kg_m_s": vec_mean.round(1),
    "max_ivt_kg_m_s": [round(float(np.nanmax(np.where(terrain_mask, np.nan, node_mag[k]))), 1)
                       for k in range(9)],
    "directional_consistency": (vec_mean / scalar_mean).round(3),
}, index=range(9)))

print(comp[["n_days", "mean_precip_mm", "pct_super_el_nino",
            "mean_ivt_kg_m_s", "max_ivt_kg_m_s", "directional_consistency"]].to_string())

w = comp["n_days"]
r_all = comp["mean_precip_mm"].corr(comp["mean_ivt_kg_m_s"])
big = comp[comp.n_days >= 9]
r_big = big["mean_precip_mm"].corr(big["mean_ivt_kg_m_s"])
print(f"\nprecip vs mean IVT, all 9 nodes        : r = {r_all:.3f}")
print(f"precip vs mean IVT, nodes with n >= 9  : r = {r_big:.3f}  (n_nodes = {len(big)})")
print(f"precip vs % super El Nino, all 9 nodes : r = "
      f"{comp['mean_precip_mm'].corr(comp['pct_super_el_nino']):.3f}")

In [ ]:
comp.to_csv(DATA_DIR / "som_node_summary_with_ivt.csv")
np.savez(DATA_DIR / "som_node_composites_ivt.npz",
         node_ivt_u=node_iu, node_ivt_v=node_iv,
         latitude=ivt.latitude.values, longitude=ivt.longitude.values)
print("saved combined node summary and IVT composites")

## Closing: what this workflow does and does not establish

**Establishes.** The heaviest Guayas rainfall days under El Niño are not a single synoptic
situation. They partition into recurring 850 hPa flow regimes that differ in moisture transport
and in mean rainfall. Crucially, the ordering of nodes by rainfall does **not** follow their
ordering by Niño 1+2 anomaly magnitude — the flow pattern carries information the SST index does
not.

**Does not establish.** Causality, and nothing about frequency change over time. The sample is
small (order 100 days, some nodes only a handful), the classification depends on choices that
section 7 of notebook 04 shows are consequential (domain especially), and daily phase
classification conflates ENSO state with the annual cycle (notebook 02, section 2).

**Natural next steps**, in rough order of value:

1. **Enlarge the sample** — select El Niño days above a percentile threshold rather than a fixed
   top-250, so the smallest nodes become interpretable. This is the single highest-value change.
2. **Add a second predictor field** to the SOM input — moisture flux convergence, or 850 hPa
   wind *plus* column humidity — and check whether nodes separate more cleanly.
3. **Composite an independent variable** by node (soil moisture, an intraseasonal index) as a
   further out-of-sample test of the classification.
4. **Downscale a representative day per node** with a convection-permitting model to test whether
   the node's large-scale flow actually produces its rainfall pattern.